In [1]:
%pip install -q sentence-transformers psycopg2-binary pgvector pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import psycopg2
import pandas as pd

from sentence_transformers import SentenceTransformer
from pgvector.psycopg2 import register_vector

D:\konst\PycharmProjects\mirea_real\Meal_planner\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="Meal_planner",
    user="postgres",
    password="postgres"
)

register_vector(conn)

In [4]:
with conn.cursor() as cur:
    cur.execute("SELECT version();")
    print(cur.fetchone()[0])

PostgreSQL 18.2 on x86_64-windows, compiled by msvc-19.44.35222, 64-bit


In [5]:
meals = pd.read_sql(
    """
    SELECT
        id,
        name,
        meal_type,
        calories,
        protein,
        fat,
        carbs,
        diet_type,
        allergens,
        tags,
        rag_text
    FROM meal_rag_data
    ORDER BY meal_type, name
    """,
    conn
)

meals.head()

C:\Users\valentin\AppData\Local\Temp\ipykernel_9564\2400362408.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  meals = pd.read_sql(


,id,name,meal_type,calories,protein,fat,carbs,diet_type,allergens,tags,rag_text
0,72f01681-78b4-41d0-b61f-398fffe80ecb,Авокадо-тост с яйцом,breakfast,390.0,18.0,20.0,34.0,vegetarian,"[яйца, глютен]","[сытный, быстрый, завтрак]",Авокадо-тост с яйцом. Цельнозерновой хлеб с ав...
1,7c6b204c-bd4a-4c96-80be-f922336e41c9,Овсянка с бананом и ягодами,breakfast,410.0,16.0,9.0,68.0,vegetarian,[молоко],"[быстрый, сладкий, высокоуглеводный, завтрак]",Овсянка с бананом и ягодами. Овсяная каша на м...
2,2fe393b5-f5f2-4500-a6be-0f139b60bccc,Омлет с овощами,breakfast,320.0,23.0,18.0,15.0,vegetarian,[яйца],"[белковый, овощи, низкоуглеводный, завтрак]","Омлет с овощами. Омлет из яиц со шпинатом, пом..."
3,78f16a38-0f67-4abe-9fdf-33d18eb3ac44,Творог с ягодами,breakfast,290.0,31.0,10.0,16.0,vegetarian,[молоко],"[белковый, быстрый, низкокалорийный, завтрак]",Творог с ягодами. Творог с греческим йогуртом ...
4,b128909b-238e-4191-8141-f46f06b9347f,Говядина с гречкой,dinner,610.0,48.0,20.0,55.0,omnivore,[],"[железо, белковый, сытный, ужин]",Говядина с гречкой. Постная говядина с гречкой...


In [6]:
print(f"Блюд: {len(meals)}")

Блюд: 17


In [7]:
print(meals.loc[0, "rag_text"])

Авокадо-тост с яйцом. Цельнозерновой хлеб с авокадо и яйцом. Тип: breakfast. Калории: 390.00. Белки: 18.00 г. Жиры: 20.00 г. Углеводы: 34.00 г. Тип питания: vegetarian. Теги: сытный, быстрый, завтрак. Ингредиенты: Авокадо 80.00 г, Помидор 80.00 г, Хлеб цельнозерновой 60.00 г, Яйцо 2.00 шт


In [8]:
model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1470.45it/s]


In [9]:
embedding = model.encode("блюдо для похудения с высоким содержанием белка")

print(len(embedding))

384


In [10]:
embeddings = model.encode(
    meals["rag_text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.03it/s]


In [11]:
print(embeddings.shape)

(17, 384)


In [12]:
embeddings[0][:10]

array([-0.04188844, -0.05540285,  0.03945709,  0.03706061, -0.00329866,
        0.00952285,  0.04537188, -0.02359592,  0.02736167, -0.04897454],
      dtype=float32)

In [13]:
with conn.cursor() as cur:
    for meal_id, embedding in zip(meals["id"], embeddings):
        cur.execute(
            """
            UPDATE meals
            SET embedding = %s
            WHERE id = %s
            """,
            (embedding, meal_id)
        )

conn.commit()

In [14]:
pd.read_sql(
    """
    SELECT
        COUNT(*) AS total,
        COUNT(embedding) AS with_embedding
    FROM meals
    """,
    conn
)

C:\Users\valentin\AppData\Local\Temp\ipykernel_9564\1738806380.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(


,total,with_embedding
0,17,17


In [15]:
def retrieve(query, limit=5):
    query_embedding = model.encode(
        query,
        normalize_embeddings=True
    )

    sql = """
        SELECT
            id,
            name,
            meal_type,
            calories,
            protein,
            fat,
            carbs,
            diet_type,
            allergens,
            tags,
            1 - (embedding <=> %s::vector) AS similarity
        FROM meals
        WHERE status = 'published'
          AND embedding IS NOT NULL
        ORDER BY embedding <=> %s::vector
        LIMIT %s
    """

    return pd.read_sql(
        sql,
        conn,
        params=(
            query_embedding,
            query_embedding,
            limit
        )
    )

In [16]:
results = retrieve(
    "хочу белковый обед для похудения",
    limit=5
)

results

C:\Users\valentin\AppData\Local\Temp\ipykernel_9564\2672450439.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(


,id,name,meal_type,calories,protein,fat,carbs,diet_type,allergens,tags,similarity
0,b51c992c-6edd-4a7b-b63b-aa40d37d6cd3,Цельнозерновой тост с творогом,snack,250.0,21.0,7.0,29.0,vegetarian,"[молоко, глютен]","[быстрый, белковый, перекус]",0.631917
1,292eb2ce-45eb-454a-a849-dc77b83a6d6a,Гречка с курицей,lunch,560.0,52.0,12.0,58.0,omnivore,[],"[белковый, сытный, для похудения, обед]",0.629105
2,c62b3479-8d1c-4650-b930-dbeeabef5c8d,Курица с рисом и брокколи,lunch,590.0,55.0,10.0,72.0,omnivore,[],"[белковый, спорт, обед, для набора]",0.604962
3,8aa58ab8-2b48-43e9-b106-fc5062753204,Индейка с киноа,lunch,520.0,48.0,13.0,50.0,omnivore,[],"[белковый, сбалансированный, обед]",0.600833
4,b128909b-238e-4191-8141-f46f06b9347f,Говядина с гречкой,dinner,610.0,48.0,20.0,55.0,omnivore,[],"[железо, белковый, сытный, ужин]",0.599687


In [17]:
retrieve(
    "веганский обед с большим количеством белка",
    limit=5
)

C:\Users\valentin\AppData\Local\Temp\ipykernel_9564\2672450439.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(


,id,name,meal_type,calories,protein,fat,carbs,diet_type,allergens,tags,similarity
0,292eb2ce-45eb-454a-a849-dc77b83a6d6a,Гречка с курицей,lunch,560.0,52.0,12.0,58.0,omnivore,[],"[белковый, сытный, для похудения, обед]",0.675829
1,b128909b-238e-4191-8141-f46f06b9347f,Говядина с гречкой,dinner,610.0,48.0,20.0,55.0,omnivore,[],"[железо, белковый, сытный, ужин]",0.673631
2,c62b3479-8d1c-4650-b930-dbeeabef5c8d,Курица с рисом и брокколи,lunch,590.0,55.0,10.0,72.0,omnivore,[],"[белковый, спорт, обед, для набора]",0.639312
3,8aa58ab8-2b48-43e9-b106-fc5062753204,Индейка с киноа,lunch,520.0,48.0,13.0,50.0,omnivore,[],"[белковый, сбалансированный, обед]",0.623651
4,b51c992c-6edd-4a7b-b63b-aa40d37d6cd3,Цельнозерновой тост с творогом,snack,250.0,21.0,7.0,29.0,vegetarian,"[молоко, глютен]","[быстрый, белковый, перекус]",0.622570


In [18]:
retrieve(
    "полезный завтрак с высоким содержанием белка",
    limit=5
)

C:\Users\valentin\AppData\Local\Temp\ipykernel_9564\2672450439.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(


,id,name,meal_type,calories,protein,fat,carbs,diet_type,allergens,tags,similarity
0,72f01681-78b4-41d0-b61f-398fffe80ecb,Авокадо-тост с яйцом,breakfast,390.0,18.0,20.0,34.0,vegetarian,"[яйца, глютен]","[сытный, быстрый, завтрак]",0.686276
1,2fe393b5-f5f2-4500-a6be-0f139b60bccc,Омлет с овощами,breakfast,320.0,23.0,18.0,15.0,vegetarian,[яйца],"[белковый, овощи, низкоуглеводный, завтрак]",0.639116
2,78f16a38-0f67-4abe-9fdf-33d18eb3ac44,Творог с ягодами,breakfast,290.0,31.0,10.0,16.0,vegetarian,[молоко],"[белковый, быстрый, низкокалорийный, завтрак]",0.621222
3,b51c992c-6edd-4a7b-b63b-aa40d37d6cd3,Цельнозерновой тост с творогом,snack,250.0,21.0,7.0,29.0,vegetarian,"[молоко, глютен]","[быстрый, белковый, перекус]",0.582546
4,7c6b204c-bd4a-4c96-80be-f922336e41c9,Овсянка с бананом и ягодами,breakfast,410.0,16.0,9.0,68.0,vegetarian,[молоко],"[быстрый, сладкий, высокоуглеводный, завтрак]",0.578014


In [19]:
retrieve(
    "что-нибудь сладкое и быстрое на завтрак",
    limit=5
)

C:\Users\valentin\AppData\Local\Temp\ipykernel_9564\2672450439.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(


,id,name,meal_type,calories,protein,fat,carbs,diet_type,allergens,tags,similarity
0,72f01681-78b4-41d0-b61f-398fffe80ecb,Авокадо-тост с яйцом,breakfast,390.0,18.0,20.0,34.0,vegetarian,"[яйца, глютен]","[сытный, быстрый, завтрак]",0.670905
1,2fe393b5-f5f2-4500-a6be-0f139b60bccc,Омлет с овощами,breakfast,320.0,23.0,18.0,15.0,vegetarian,[яйца],"[белковый, овощи, низкоуглеводный, завтрак]",0.604371
2,78f16a38-0f67-4abe-9fdf-33d18eb3ac44,Творог с ягодами,breakfast,290.0,31.0,10.0,16.0,vegetarian,[молоко],"[белковый, быстрый, низкокалорийный, завтрак]",0.601343
3,7c6b204c-bd4a-4c96-80be-f922336e41c9,Овсянка с бананом и ягодами,breakfast,410.0,16.0,9.0,68.0,vegetarian,[молоко],"[быстрый, сладкий, высокоуглеводный, завтрак]",0.556992
4,b51c992c-6edd-4a7b-b63b-aa40d37d6cd3,Цельнозерновой тост с творогом,snack,250.0,21.0,7.0,29.0,vegetarian,"[молоко, глютен]","[быстрый, белковый, перекус]",0.538829


In [20]:
def retrieve_by_meal_type(query, meal_type, limit=5):
    query_embedding = model.encode(
        query,
        normalize_embeddings=True
    )

    sql = """
        SELECT
            id,
            name,
            meal_type,
            calories,
            protein,
            fat,
            carbs,
            diet_type,
            allergens,
            tags,
            1 - (embedding <=> %s::vector) AS similarity
        FROM meals
        WHERE status = 'published'
          AND embedding IS NOT NULL
          AND meal_type = %s
        ORDER BY embedding <=> %s::vector
        LIMIT %s
    """

    return pd.read_sql(
        sql,
        conn,
        params=(
            query_embedding,
            meal_type,
            query_embedding,
            limit
        )
    )

In [21]:
retrieve_by_meal_type(
    "белковое блюдо для похудения",
    "lunch",
    5
)

C:\Users\valentin\AppData\Local\Temp\ipykernel_9564\3881925900.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(


,id,name,meal_type,calories,protein,fat,carbs,diet_type,allergens,tags,similarity
0,292eb2ce-45eb-454a-a849-dc77b83a6d6a,Гречка с курицей,lunch,560.0,52.0,12.0,58.0,omnivore,[],"[белковый, сытный, для похудения, обед]",0.581444
1,8aa58ab8-2b48-43e9-b106-fc5062753204,Индейка с киноа,lunch,520.0,48.0,13.0,50.0,omnivore,[],"[белковый, сбалансированный, обед]",0.559957
2,c62b3479-8d1c-4650-b930-dbeeabef5c8d,Курица с рисом и брокколи,lunch,590.0,55.0,10.0,72.0,omnivore,[],"[белковый, спорт, обед, для набора]",0.548368
3,062be53a-1329-466c-8415-1a84e3159cae,Нут с овощами и киноа,lunch,510.0,21.0,15.0,70.0,vegan,[],"[веганский, растительный белок, клетчатка, обед]",0.539599
4,86e7d145-868e-4239-a97c-bf37a8a88011,Чечевица с овощами,lunch,430.0,24.0,9.0,58.0,vegan,[],"[растительный белок, веганский, клетчатка, обед]",0.533361


In [22]:
def retrieve_filtered(
    query,
    meal_type=None,
    diet_types=None,
    limit=5
):
    query_embedding = model.encode(
        query,
        normalize_embeddings=True
    )

    conditions = [
        "status = 'published'",
        "embedding IS NOT NULL"
    ]

    params = [query_embedding]

    if meal_type:
        conditions.append("meal_type = %s")
        params.append(meal_type)

    if diet_types:
        placeholders = ", ".join(["%s"] * len(diet_types))
        conditions.append(f"diet_type IN ({placeholders})")
        params.extend(diet_types)

    params.append(query_embedding)
    params.append(limit)

    sql = f"""
        SELECT
            id,
            name,
            meal_type,
            calories,
            protein,
            fat,
            carbs,
            diet_type,
            allergens,
            tags,
            1 - (embedding <=> %s::vector) AS similarity
        FROM meals
        WHERE {" AND ".join(conditions)}
        ORDER BY embedding <=> %s::vector
        LIMIT %s
    """

    return pd.read_sql(
        sql,
        conn,
        params=params
    )

In [23]:
retrieve_filtered(
    "белковый сытный обед",
    meal_type="lunch",
    diet_types=["vegetarian", "vegan"],
    limit=5
)

C:\Users\valentin\AppData\Local\Temp\ipykernel_9564\2391198150.py:50: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(


,id,name,meal_type,calories,protein,fat,carbs,diet_type,allergens,tags,similarity
0,062be53a-1329-466c-8415-1a84e3159cae,Нут с овощами и киноа,lunch,510.0,21.0,15.0,70.0,vegan,[],"[веганский, растительный белок, клетчатка, обед]",0.560088
1,86e7d145-868e-4239-a97c-bf37a8a88011,Чечевица с овощами,lunch,430.0,24.0,9.0,58.0,vegan,[],"[растительный белок, веганский, клетчатка, обед]",0.542171


In [24]:
retrieve_filtered(
    "белковый сытный обед",
    meal_type="lunch",
    diet_types=["vegan"],
    limit=5
)

C:\Users\valentin\AppData\Local\Temp\ipykernel_9564\2391198150.py:50: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(


,id,name,meal_type,calories,protein,fat,carbs,diet_type,allergens,tags,similarity
0,062be53a-1329-466c-8415-1a84e3159cae,Нут с овощами и киноа,lunch,510.0,21.0,15.0,70.0,vegan,[],"[веганский, растительный белок, клетчатка, обед]",0.560088
1,86e7d145-868e-4239-a97c-bf37a8a88011,Чечевица с овощами,lunch,430.0,24.0,9.0,58.0,vegan,[],"[растительный белок, веганский, клетчатка, обед]",0.542171


In [25]:
ivan = pd.read_sql(
    """
    SELECT
        u.id,
        u.name,
        p.goal,
        p.daily_calories,
        p.protein_target,
        p.fat_target,
        p.carbs_target,
        p.budget,
        p.diet_type,
        p.restrictions
    FROM users u
    JOIN profiles p ON p.user_id = u.id
    WHERE u.email = 'ivan@example.com'
    """,
    conn
)

ivan

C:\Users\valentin\AppData\Local\Temp\ipykernel_9564\847040346.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  ivan = pd.read_sql(


,id,name,goal,daily_calories,protein_target,fat_target,carbs_target,budget,diet_type,restrictions
0,0d3ef8c9-4b8b-4fb8-ac63-629a0f3f9c40,Иван,lose_weight,1800,130.0,60.0,180.0,5000.0,omnivore,без арахиса


In [26]:
ivan_query = f"""
Цель: {ivan.loc[0, 'goal']}.
Калорийность: {ivan.loc[0, 'daily_calories']} ккал.
Белок: {ivan.loc[0, 'protein_target']} г.
Жиры: {ivan.loc[0, 'fat_target']} г.
Углеводы: {ivan.loc[0, 'carbs_target']} г.
Тип питания: {ivan.loc[0, 'diet_type']}.
Ограничения: {ivan.loc[0, 'restrictions']}.
Нужны подходящие блюда для рациона.
"""

print(ivan_query)


Цель: lose_weight.
Калорийность: 1800 ккал.
Белок: 130.0 г.
Жиры: 60.0 г.
Углеводы: 180.0 г.
Тип питания: omnivore.
Ограничения: без арахиса.
Нужны подходящие блюда для рациона.



In [27]:
ivan_results = retrieve_filtered(
    ivan_query,
    diet_types=["omnivore", "vegetarian", "vegan"],
    limit=10
)

ivan_results

C:\Users\valentin\AppData\Local\Temp\ipykernel_9564\2391198150.py:50: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(


,id,name,meal_type,calories,protein,fat,carbs,diet_type,allergens,tags,similarity
0,8aa58ab8-2b48-43e9-b106-fc5062753204,Индейка с киноа,lunch,520.0,48.0,13.0,50.0,omnivore,[],"[белковый, сбалансированный, обед]",0.627516
1,292eb2ce-45eb-454a-a849-dc77b83a6d6a,Гречка с курицей,lunch,560.0,52.0,12.0,58.0,omnivore,[],"[белковый, сытный, для похудения, обед]",0.571124
2,b51c992c-6edd-4a7b-b63b-aa40d37d6cd3,Цельнозерновой тост с творогом,snack,250.0,21.0,7.0,29.0,vegetarian,"[молоко, глютен]","[быстрый, белковый, перекус]",0.567688
3,42880606-eb74-4ebf-941d-cba12b677620,Индейка с овощами,dinner,410.0,48.0,12.0,24.0,omnivore,[],"[белковый, низкокалорийный, для похудения, ужин]",0.550088
4,b128909b-238e-4191-8141-f46f06b9347f,Говядина с гречкой,dinner,610.0,48.0,20.0,55.0,omnivore,[],"[железо, белковый, сытный, ужин]",0.539456
5,c62b3479-8d1c-4650-b930-dbeeabef5c8d,Курица с рисом и брокколи,lunch,590.0,55.0,10.0,72.0,omnivore,[],"[белковый, спорт, обед, для набора]",0.539017
6,94141d11-7028-4617-aac6-9220c7794003,Хумус с овощами,snack,240.0,9.0,11.0,28.0,vegan,[кунжут],"[веганский, клетчатка, перекус]",0.533344
7,062be53a-1329-466c-8415-1a84e3159cae,Нут с овощами и киноа,lunch,510.0,21.0,15.0,70.0,vegan,[],"[веганский, растительный белок, клетчатка, обед]",0.531945
8,fa753550-aec2-4d24-b76a-d92e84a125c8,Творог с яблоком,snack,260.0,28.0,7.0,22.0,vegetarian,[молоко],"[белковый, быстрый, перекус]",0.530544
9,7c6b204c-bd4a-4c96-80be-f922336e41c9,Овсянка с бананом и ягодами,breakfast,410.0,16.0,9.0,68.0,vegetarian,[молоко],"[быстрый, сладкий, высокоуглеводный, завтрак]",0.523987


In [28]:
def build_context(results):
    context = []

    for _, row in results.iterrows():
        context.append(
            f"""
meal_id: {row['id']}
name: {row['name']}
type: {row['meal_type']}
calories: {row['calories']}
protein: {row['protein']} g
fat: {row['fat']} g
carbs: {row['carbs']} g
diet: {row['diet_type']}
tags: {', '.join(row['tags'])}
similarity: {row['similarity']:.4f}
"""
        )

    return "\n".join(context)

In [29]:
context = build_context(ivan_results)

print(context)


meal_id: 8aa58ab8-2b48-43e9-b106-fc5062753204
name: Индейка с киноа
type: lunch
calories: 520.0
protein: 48.0 g
fat: 13.0 g
carbs: 50.0 g
diet: omnivore
tags: белковый, сбалансированный, обед
similarity: 0.6275


meal_id: 292eb2ce-45eb-454a-a849-dc77b83a6d6a
name: Гречка с курицей
type: lunch
calories: 560.0
protein: 52.0 g
fat: 12.0 g
carbs: 58.0 g
diet: omnivore
tags: белковый, сытный, для похудения, обед
similarity: 0.5711


meal_id: b51c992c-6edd-4a7b-b63b-aa40d37d6cd3
name: Цельнозерновой тост с творогом
type: snack
calories: 250.0
protein: 21.0 g
fat: 7.0 g
carbs: 29.0 g
diet: vegetarian
tags: быстрый, белковый, перекус
similarity: 0.5677


meal_id: 42880606-eb74-4ebf-941d-cba12b677620
name: Индейка с овощами
type: dinner
calories: 410.0
protein: 48.0 g
fat: 12.0 g
carbs: 24.0 g
diet: omnivore
tags: белковый, низкокалорийный, для похудения, ужин
similarity: 0.5501


meal_id: b128909b-238e-4191-8141-f46f06b9347f
name: Говядина с гречкой
type: dinner
calories: 610.0
protein: 48.0 